## Final Results

| Model type | Training images | Best epoch | Validation accuracy at best epoch | Test accuracy |
|---|---:|---:|---:|---:|
| From scratch | 1,000 | 38 | 75.800% | 74.600% |
| From scratch | 1,500 | 46 | 82.400% | 81.400% |
| From scratch | 2,000 | 47 | 85.400% | 82.600% |
| Pretrained VGG16 | 1,000 | 25 | 99.000% | 97.800% |
| Pretrained VGG16 | 1,500 | 11 | 98.800% | 97.000% |
| Pretrained VGG16 | 2,000 | 30 | 99.000% | 97.600% |

### Summary

The pretrained VGG16 models achieved substantially higher validation and test accuracy than the models trained from scratch. The custom CNN's test accuracy improved from 74.6% to 82.6% as the training set increased from 1,000 to 2,000 images. The pretrained models achieved test accuracies between 97.0% and 97.8%, with the model trained on 1,000 images producing the highest test accuracy. Overall accuarcy between the three pretrained models is all within .8%, which could be due to ordinary sampling variation. All models were optimized using dropout and data augmentation techniques. 

In [6]:
from pathlib import Path

project_dir = Path.home() / "ml" / "cats_dogs"
print("Project folder exists:", project_dir.exists())
print("Files in folder:")
for item in project_dir.iterdir():
    print(item.name)

Project folder exists: True
Files in folder:
cats_dogs_assignment
cats_vs_dogs_small.zip
cats_vs_dogs_small.zip:Zone.Identifier
.ipynb_checkpoints
Untitled.ipynb
cats_vs_dogs_small


In [7]:
import os
import shutil
import pathlib
import zipfile

project_dir = pathlib.Path.home() / "ml" / "cats_dogs"
zip_path = project_dir / "cats_vs_dogs_small.zip"
original_dir = project_dir / "cats_vs_dogs_small"
new_base_dir = project_dir / "cats_dogs_assignment"

In [8]:
from pathlib import Path
import zipfile

project_dir = Path.home() / "ml" / "cats_dogs"
zip_path = project_dir / "cats_vs_dogs_small.zip"
original_dir = project_dir / "cats_vs_dogs_small"

if not original_dir.exists():
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(project_dir)

print("Extraction complete.")
print("Project contents:", [p.name for p in project_dir.iterdir()])

Extraction complete.
Project contents: ['cats_dogs_assignment', 'cats_vs_dogs_small.zip', 'cats_vs_dogs_small.zip:Zone.Identifier', '.ipynb_checkpoints', 'Untitled.ipynb', 'cats_vs_dogs_small']


In [9]:
for path in sorted(original_dir.iterdir()):
    if path.is_dir():
        print(f"\n{path.name}/")
        for subdir in sorted(path.iterdir()):
            if subdir.is_dir():
                print(f"  {subdir.name}: {len(list(subdir.glob('*.jpg')))} JPG images")


test/
  cats: 500 JPG images
  dogs: 500 JPG images

train/
  cats: 1000 JPG images
  dogs: 1000 JPG images

validation/
  cats: 500 JPG images
  dogs: 500 JPG images


In [10]:
import pathlib
import shutil

project_dir = pathlib.Path.home() / "ml" / "cats_dogs"
original_dir = project_dir / "cats_vs_dogs_small"
new_base_dir = project_dir / "cats_dogs_assignment"

def make_subset(source_split, subset_name, end_index):
    for category in ("cats", "dogs"):
        source_dir = original_dir / source_split / category
        destination_dir = new_base_dir / subset_name / category
        destination_dir.mkdir(parents=True, exist_ok=True)

        fnames = sorted(source_dir.glob("*.jpg"))[:end_index]

        for fname in fnames:
            shutil.copy2(fname, destination_dir / fname.name)

make_subset("train", "train_1000", 500)
make_subset("train", "train_1500", 750)
make_subset("train", "train_2000", 1000)
make_subset("validation", "validation", 250)
make_subset("test", "test", 250)

print("Subsets created under:", new_base_dir)

Subsets created under: /home/claxta11/ml/cats_dogs/cats_dogs_assignment


In [11]:
for split in ("train_1000", "train_1500", "train_2000",
              "validation", "test"):
    print(f"\n{split}")
    for category in ("cats", "dogs"):
        folder = new_base_dir / split / category
        print(f"  {category}: {len(list(folder.glob('*.jpg')))} images")


train_1000
  cats: 500 images
  dogs: 500 images

train_1500
  cats: 750 images
  dogs: 750 images

train_2000
  cats: 1000 images
  dogs: 1000 images

validation
  cats: 250 images
  dogs: 250 images

test
  cats: 250 images
  dogs: 250 images


In [12]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [13]:
#instantiate model_from_scratch
from tensorflow import keras 
from tensorflow.keras import layers

#Adding data augmentation to scratch model
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.2),
    ]
)

 
#function to make the model from scratch and configure 
def make_model_from_scratch():
    inputs = keras.Input(shape=(180, 180, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)
    x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Conv2D(filters=256, kernel_size=3, activation="relu")(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

    model.compile(
        loss="binary_crossentropy",
        optimizer="rmsprop",
        metrics=["accuracy"]
    )

    return model


In [14]:

#reading images 
from tensorflow.keras.utils import image_dataset_from_directory
  
train_1000_dataset = image_dataset_from_directory(
    new_base_dir / "train_1000",
    image_size=(180, 180),
    batch_size=32
)

train_1500_dataset = image_dataset_from_directory(
    new_base_dir / "train_1500",
    image_size=(180, 180),
    batch_size=32
)

train_2000_dataset = image_dataset_from_directory(
    new_base_dir / "train_2000",
    image_size=(180, 180),
    batch_size=32
)

validation_dataset = image_dataset_from_directory(
    new_base_dir / "validation",
    image_size=(180, 180),
    batch_size=32
)

test_dataset = image_dataset_from_directory(
    new_base_dir / "test",
    image_size=(180, 180),
    batch_size=32
)


Found 1000 files belonging to 2 classes.
Found 1500 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.
Found 500 files belonging to 2 classes.
Found 500 files belonging to 2 classes.


In [15]:
#Make three distinct models
model_1000 = make_model_from_scratch()
model_1500 = make_model_from_scratch()
model_2000 = make_model_from_scratch()

#function for callbacks
def make_callbacks(training_size):
    return [
        keras.callbacks.ModelCheckpoint(
            filepath=f"convnet_from_scratch_{training_size}.keras",
            save_best_only=True,
            monitor="val_loss"
        )
    ]


In [16]:
history_1000 = model_1000.fit(
    train_1000_dataset,
    epochs=100,
    validation_data=validation_dataset,
    callbacks=make_callbacks("1000")
)

history_1500 = model_1500.fit(
    train_1500_dataset,
    epochs=100,
    validation_data=validation_dataset,
    callbacks=make_callbacks("1500")
)

history_2000 = model_2000.fit(
    train_2000_dataset,
    epochs=100,
    validation_data=validation_dataset,
    callbacks=make_callbacks("2000")
)


test_model_1000 = keras.models.load_model(
    "convnet_from_scratch_1000.keras")
test_loss_1000, test_acc_1000 = test_model_1000.evaluate(test_dataset)
print(f"1000 Test accuracy: {test_acc_1000:.3f}")

test_model_1500 = keras.models.load_model(
    "convnet_from_scratch_1500.keras")
test_loss_1500, test_acc_1500 = test_model_1500.evaluate(test_dataset)
print(f"1500 Test accuracy: {test_acc_1500:.3f}")

test_model_2000 = keras.models.load_model(
    "convnet_from_scratch_2000.keras")
test_loss_2000, test_acc_2000 = test_model_2000.evaluate(test_dataset)
print(f"2000 Test accuracy: {test_acc_2000:.3f}")


Epoch 1/100


/home/claxta11/miniconda3/envs/tensorflow/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-10-15 19:06:37.496529: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92600


32/32 ━━━━━━━━━━━━━━━━━━━━ 8s 94ms/step - accuracy: 0.5140 - loss: 0.7079 - val_accuracy: 0.5000 - val_loss: 0.6928
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.4840 - loss: 0.6935 - val_accuracy: 0.5000 - val_loss: 0.6919
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 56ms/step - accuracy: 0.5220 - loss: 0.6932 - val_accuracy: 0.5000 - val_loss: 0.6897
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - accuracy: 0.5670 - loss: 0.6900 - val_accuracy: 0.6200 - val_loss: 0.6817
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - accuracy: 0.5670 - loss: 0.6828 - val_accuracy: 0.6080 - val_loss: 0.6704
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 52ms/step - accuracy: 0.5760 - loss: 0.6846 - val_accuracy: 0.5180 - val_loss: 0.6724
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - accuracy: 0.5990 - loss: 0.6592 - val_accuracy: 0.5020 - val_loss: 0.7094
Epoch 8/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - accuracy: 0.6050 - loss: 0.6625 - val_accuracy: 0.5000 - val_l

Corrupt JPEG data: 128 extraneous bytes before marker 0xd9


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7460 - loss: 0.5466
1000 Test accuracy: 0.746
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8140 - loss: 0.4561
1500 Test accuracy: 0.814


Corrupt JPEG data: 128 extraneous bytes before marker 0xd9


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8260 - loss: 0.4190
2000 Test accuracy: 0.826


Corrupt JPEG data: 128 extraneous bytes before marker 0xd9


In [18]:

############PRETRAINED model function
def make_pretrained_model():

    conv_base = keras.applications.vgg16.VGG16(
        weights="imagenet",
        include_top=False,
        input_shape=(180, 180, 3)
    )

    conv_base.trainable = False

    data_augmentation = keras.Sequential(
        [
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.2),
        ]
    )

    inputs = keras.Input(shape=(180, 180, 3))

    x = data_augmentation(inputs)
    x = keras.applications.vgg16.preprocess_input(x)

    x = conv_base(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        loss="binary_crossentropy",
        optimizer="rmsprop",
        metrics=["accuracy"]
    )

    return model


In [20]:
#make three pretrained models 
pretrained_1000 = make_pretrained_model()
pretrained_1500 = make_pretrained_model()
pretrained_2000 = make_pretrained_model()
#function for pretrained callbacks
def pretrained_callbacks(training_size):
    return [
        keras.callbacks.ModelCheckpoint(
            filepath=f"convnet_pretrained_{training_size}.keras",
            save_best_only=True,
            monitor="val_loss"
        )
    ]


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [21]:
#fit and do callbacks
phistory_1000 = pretrained_1000.fit(
    train_1000_dataset,
    epochs=30,
    validation_data=validation_dataset,
    callbacks=pretrained_callbacks("1000")
)

phistory_1500 = pretrained_1500.fit(
    train_1500_dataset,
    epochs=30,
    validation_data=validation_dataset,
    callbacks=pretrained_callbacks("1500")
)

phistory_2000 = pretrained_2000.fit(
    train_2000_dataset,
    epochs=30,
    validation_data=validation_dataset,
    callbacks=pretrained_callbacks("2000")
)


pretrained_model_1000 = keras.models.load_model(
    "convnet_pretrained_1000.keras")
pret_loss_1000, pret_acc_1000 = pretrained_model_1000.evaluate(test_dataset)
print(f"1000 Test accuracy: {pret_acc_1000:.3f}")

pretrained_model_1500 = keras.models.load_model(
    "convnet_pretrained_1500.keras")
pret_loss_1500, pret_acc_1500 = pretrained_model_1500.evaluate(test_dataset)
print(f"1500 Test accuracy: {pret_acc_1500:.3f}")

pretrained_model_2000 = keras.models.load_model(
    "convnet_pretrained_2000.keras")
pret_loss_2000, pret_acc_2000 = pretrained_model_2000.evaluate(test_dataset)
print(f"2000 Test accuracy: {pret_acc_2000:.3f}")


Epoch 1/30


2026-10-15 19:21:49.018318: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 3.04GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-10-15 19:21:49.823710: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.55GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.8580 - loss: 10.5177

2026-10-15 19:21:57.400154: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.66GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


32/32 ━━━━━━━━━━━━━━━━━━━━ 12s 256ms/step - accuracy: 0.8580 - loss: 10.5177 - val_accuracy: 0.9460 - val_loss: 0.9273
Epoch 2/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 164ms/step - accuracy: 0.9150 - loss: 1.7837 - val_accuracy: 0.9780 - val_loss: 0.4234
Epoch 3/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 155ms/step - accuracy: 0.9470 - loss: 0.8219 - val_accuracy: 0.9620 - val_loss: 0.7019
Epoch 4/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 163ms/step - accuracy: 0.9430 - loss: 0.6666 - val_accuracy: 0.9720 - val_loss: 0.2580
Epoch 5/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 155ms/step - accuracy: 0.9550 - loss: 0.5532 - val_accuracy: 0.9720 - val_loss: 0.2924
Epoch 6/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 163ms/step - accuracy: 0.9590 - loss: 0.3437 - val_accuracy: 0.9820 - val_loss: 0.1569
Epoch 7/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 156ms/step - accuracy: 0.9600 - loss: 0.2556 - val_accuracy: 0.9780 - val_loss: 0.2301
Epoch 8/30
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 156ms/step - accuracy: 0.9770 - loss: 0.1272 - val_accuracy: 0.9860 - va

...

16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 102ms/step - accuracy: 0.9780 - loss: 0.6375
1000 Test accuracy: 0.978
 4/16 ━━━━━━━━━━━━━━━━━━━━ 1s 104ms/step - accuracy: 0.9688 - loss: 0.1555

Corrupt JPEG data: 128 extraneous bytes before marker 0xd9


16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 101ms/step - accuracy: 0.9700 - loss: 0.3069
1500 Test accuracy: 0.970
 4/16 ━━━━━━━━━━━━━━━━━━━━ 1s 102ms/step - accuracy: 0.9766 - loss: 0.4720

Corrupt JPEG data: 128 extraneous bytes before marker 0xd9


16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 101ms/step - accuracy: 0.9760 - loss: 0.4201
2000 Test accuracy: 0.976


In [22]:
import numpy as np

#this code stores variables to show the best epoch for each model
#scratch models

best_epoch_1000 = np.argmin(history_1000.history["val_loss"]) + 1
best_val_acc_1000 = history_1000.history["val_accuracy"][best_epoch_1000 - 1]

best_epoch_1500 = np.argmin(history_1500.history["val_loss"]) + 1
best_val_acc_1500 = history_1500.history["val_accuracy"][best_epoch_1500 - 1]

best_epoch_2000 = np.argmin(history_2000.history["val_loss"]) + 1
best_val_acc_2000 = history_2000.history["val_accuracy"][best_epoch_2000 - 1]


#Pretrained models

best_p_epoch_1000 = np.argmin(phistory_1000.history["val_loss"]) + 1
best_p_val_acc_1000 = phistory_1000.history["val_accuracy"][best_p_epoch_1000 - 1]

best_p_epoch_1500 = np.argmin(phistory_1500.history["val_loss"]) + 1
best_p_val_acc_1500 = phistory_1500.history["val_accuracy"][best_p_epoch_1500 - 1]

best_p_epoch_2000 = np.argmin(phistory_2000.history["val_loss"]) + 1
best_p_val_acc_2000 = phistory_2000.history["val_accuracy"][best_p_epoch_2000 - 1]


In [26]:
#Tabular Final Results – EXECUTE LAST- then copy paste the table up top
%pip install pandas
import pandas as pd
from IPython.display import display

results_table = pd.DataFrame(
    {
        "Model type": [
            "From scratch",
            "From scratch",
            "From scratch",
            "Pretrained VGG16",
            "Pretrained VGG16",
            "Pretrained VGG16",
        ],
        "Training images": [
            1000,
            1500,
            2000,
            1000,
            1500,
            2000,
        ],
        "Best epoch": [
            best_epoch_1000,
            best_epoch_1500,
            best_epoch_2000,
            best_p_epoch_1000,
            best_p_epoch_1500,
            best_p_epoch_2000,
        ],
        "Validation accuracy at best epoch": [
            best_val_acc_1000,
            best_val_acc_1500,
            best_val_acc_2000,
            best_p_val_acc_1000,
            best_p_val_acc_1500,
            best_p_val_acc_2000,
        ],
        "Test accuracy": [
            test_acc_1000,
            test_acc_1500,
            test_acc_2000,
            pret_acc_1000,
            pret_acc_1500,
            pret_acc_2000,
        ],
    }
)

display(
    results_table.style.format(
        {
            "Validation accuracy at best epoch": "{:.3%}",
            "Test accuracy": "{:.3%}",
        }
    ))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 37.7 MB/s  0:00:00eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


,Model type,Training images,Best epoch,Validation accuracy at best epoch,Test accuracy
0,From scratch,1000,38,75.800%,74.600%
1,From scratch,1500,46,82.400%,81.400%
2,From scratch,2000,47,85.400%,82.600%
3,Pretrained VGG16,1000,25,99.000%,97.800%
4,Pretrained VGG16,1500,11,98.800%,97.000%
5,Pretrained VGG16,2000,30,99.000%,97.600%
